# Colab Training - Hybrid BiLSTM POS Tagger

Notebook ini menyiapkan pipeline end-to-end:
1. Mount Google Drive
2. Clone repository (opsional)
3. Konversi `.conllu` -> `.jsonl`
4. Training model
5. Lihat metrik `last.pt` dan `best.pt`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import json
from pathlib import Path

# Ubah sesuai struktur Drive kamu
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_PARENT = DRIVE_ROOT / 'projects'     # contoh: MyDrive/projects
REPO_URL = 'https://github.com/NazirArifin/pantunify.git'
REPO_DIR = PROJECT_PARENT / 'pantunify'

PROJECT_PARENT.mkdir(parents=True, exist_ok=True)
print('PROJECT_PARENT =', PROJECT_PARENT)
print('REPO_DIR =', REPO_DIR)


In [ ]:
import subprocess

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repo sudah ada, skip clone.')

%cd {REPO_DIR}
!git --no-pager status -sb


In [ ]:
!python -V
!pip -q install --upgrade pip
!pip -q install torch


## Konfigurasi path dataset

Sesuaikan jika lokasi dataset kamu berbeda.

In [ ]:
DATA_DIR = REPO_DIR / 'data' / 'article5'
TRAIN_GSD = DATA_DIR / 'UD_Indonesian-GSD' / 'id_gsd-ud-train.conllu'
DEV_GSD = DATA_DIR / 'UD_Indonesian-GSD' / 'id_gsd-ud-dev.conllu'
TRAIN_CSUI = DATA_DIR / 'id_csui-ud-train.conllu'

OUT_DIR = DATA_DIR / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [TRAIN_GSD, DEV_GSD, TRAIN_CSUI]:
    print(p, '->', 'OK' if p.exists() else 'MISSING')


In [ ]:
CONVERTER = REPO_DIR / 'scripts' / 'article5' / 'convert_conllu_to_jsonl.py'
assert CONVERTER.exists(), f'Converter tidak ditemukan: {CONVERTER}'

!python {CONVERTER} \
  --train-conllu {TRAIN_GSD} {TRAIN_CSUI} \
  --dev-conllu {DEV_GSD} \
  --out-dir {OUT_DIR}


In [ ]:
VOCAB_PATH = OUT_DIR / 'vocab.json'
with open(VOCAB_PATH, 'r', encoding='utf-8') as f:
    vocab = json.load(f)

num_chars = len(vocab['char_vocab'])
vocab_size = len(vocab['word_vocab'])
num_classes = len(vocab['tag_vocab'])

print('num_chars  =', num_chars)
print('vocab_size =', vocab_size)
print('num_classes=', num_classes)


In [ ]:
TRAIN_SCRIPT = REPO_DIR / 'scripts' / 'article5' / 'train_hybrid_bilstm_pos.py'
CKPT_DIR = REPO_DIR / 'scripts' / 'article5' / 'checkpoints_colab'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_JSONL = OUT_DIR / 'train.jsonl'
DEV_JSONL = OUT_DIR / 'dev.jsonl'

!python {TRAIN_SCRIPT} \
  --train-jsonl {TRAIN_JSONL} \
  --valid-jsonl {DEV_JSONL} \
  --num-chars {num_chars} \
  --vocab-size {vocab_size} \
  --num-classes {num_classes} \
  --save-dir {CKPT_DIR} \
  --epochs 20 \
  --batch-size 64 \
  --word-emb-dim 300


In [ ]:
import torch

last_ckpt = CKPT_DIR / 'last.pt'
best_ckpt = CKPT_DIR / 'best.pt'

print('last exists:', last_ckpt.exists())
print('best exists:', best_ckpt.exists())

if last_ckpt.exists():
    last_obj = torch.load(last_ckpt, map_location='cpu')
    print('\n[LAST] epoch:', last_obj.get('epoch'))
    print('[LAST] metrics:', last_obj.get('metrics'))

if best_ckpt.exists():
    best_obj = torch.load(best_ckpt, map_location='cpu')
    print('\n[BEST] epoch:', best_obj.get('epoch'))
    print('[BEST] metrics:', best_obj.get('metrics'))
